# Cone-Projected Greedy — walkthrough

Benaceur, Ern, Ehrlacher, [`hal-02081485v2`](https://hal.science/hal-02081485v2).

Read `REPRODUCTION_NOTES.md` first. In particular: the 2-D contact application of
§3 and §6 is **not** implemented, and the numbers below reproduce nothing in §6.

Runs on CPU in a few seconds.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

# The cone algorithms live in the shared library ~/rb_vi_shared, together with
# the sequel in ~/stable_model_reduction_vi. _shared_path puts it on sys.path.
# Citations in shared code carry a paper tag -- [BEE20] is this paper (hal
# preprint v2 numbering), [NDEE22] the sequel -- because the two papers number
# "Algorithm 1", "Algorithm 2" and "Eq. (57)" differently. See
# ~/rb_vi_shared/README.md.
import _shared_path  # noqa: F401
from rb_vi_common import cone_projected_greedy, project_onto_cone, norm_Lambda

from toy_problem import generate_snapshots, obstacle_gap, solve_hf
from nmf_baseline import nmf
from rb_online import pod_basis, solve_reduced, pga_enrichment, online_enrichment

## 1. The multiplier snapshots

§2 sets up the saddle-point problem (Eq. 5–6) whose discrete form is Eq. (9).
Its dual solution `λ(µ)` is the contact pressure: **non-negative**, and supported
only where the constraint is active. The active set moves with `µ` — that is what
makes the dual manifold hard to compress.

In [ ]:
S_pri, S_du, params, A, F = generate_snapshots(N=60, n_train=40)
print("dual snapshots:", S_du.shape, " min entry:", S_du.min())

fig, ax = plt.subplots(figsize=(7, 3.2))
for j in range(0, S_du.shape[1], 5):
    ax.plot(S_du[:, j], lw=1.2, label=f"mu={params[j,0]:.2f},{params[j,1]:.2f}")
ax.set_title(r"Lagrange multiplier snapshots $\lambda(\mu)$ — all $\geq 0$")
ax.set_xlabel("contact node"); ax.legend(fontsize=7)
plt.tight_layout()

## 2. Why not POD

§5: *"its spanning vectors should all have non-negative components. Consequently,
the POD is not appropriate to build $\hat W_R^+$."*

Check it directly — POD modes of a non-negative snapshot set still carry mixed signs.

In [ ]:
pod_du = pod_basis(S_du, n_modes=5)
for k in range(5):
    m = pod_du[:, k]
    print(f"POD mode {k}: min={m.min():+.4f}  max={m.max():+.4f}  "
          f"sign-mixed={m.min() < 0 < m.max()}")

Every mode mixes signs, so a POD basis cannot represent the dual cone without
admitting negative pressures. Hence Algorithm 2.

## 3. The cone projection $\Pi_{K^+}$ — Eq. (57)

The numerical core. `min_{c ≥ 0} ‖λ − Gc‖` is a non-negative least squares problem.
It is a projection onto a **cone**, not a subspace, so it is not linear in `λ`:
there is no projection matrix. Verify the two defining properties.

In [ ]:
G = S_du[:, :6]
lam = S_du[:, 20]
proj, c = project_onto_cone(lam, G)

print("coefficients c:", np.round(c, 4))
print("all c >= 0:", bool((c >= 0).all()))
print("projection is non-negative:", bool((proj >= -1e-12).all()))

# Non-linearity: projecting a scaled input is NOT the scaled projection in general
# once the active set of the NNLS changes.
p2, _ = project_onto_cone(lam - 0.9 * G[:, 0], G)
print("\nlinear?", np.allclose(p2, proj - 0.9 * G[:, 0]))

## 4. Algorithm 2

The tolerance `ε_du` in Eq. (58) is **absolute**, so it only means something
relative to the snapshot scale. Sweep it.

In [ ]:
scale = max(norm_Lambda(S_du[:, j]) for j in range(S_du.shape[1]))
print(f"max ||lambda(mu)|| = {scale:.2f}\n")
for frac in (0.5, 0.3, 0.2, 0.1, 0.05):
    r = cone_projected_greedy(S_du, eps_du=frac * scale)
    print(f"  eps_du = {frac:4.2f} * scale = {frac*scale:8.3f}  ->  R = {r.R}")

In [ ]:
res = cone_projected_greedy(S_du, eps_du=1e-12, max_R=10)
print("selected mu indices:", res.selected_indices)

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.semilogy(range(1, len(res.residuals) + 1), res.residuals, "o-")
ax.set_xlabel("R"); ax.set_ylabel(r"$r_n$, Eq. (58)")
ax.set_title("Greedy residual — monotone, hence hierarchical")
ax.grid(alpha=.3); plt.tight_layout()

## 5. CPG vs NMF at matched cardinality

§5 offers NMF (from [2]) as the alternative and criticizes it for taking a
**cardinality** rather than a **tolerance**. Compare at equal `R`.

In [ ]:
def cone_error(S, gens):
    return max(norm_Lambda(S[:, j] - project_onto_cone(S[:, j], gens)[0])
               for j in range(S.shape[1]))

print(f"CPG                 : {cone_error(S_du, res.generators):.4e}")
for seed in (0, 1, 2):
    print(f"NMF (seed={seed})        : {cone_error(S_du, nmf(S_du, R=res.R, seed=seed)):.4e}")

**NMF wins on this toy.** That is expected and is not a refutation of the paper:
CPG generators are *selected snapshots*, NMF atoms are *optimized* and free to sit
anywhere in the non-negative orthant, so at equal `R` the freer parametrization
should usually fit better.

The properties §5 actually claims for CPG are visible above instead: it is
tolerance-driven (cell 4), hierarchical (monotone residual), and deterministic —
note the spread across NMF seeds, which §6.4 predicts by observing that the
factorization is non-unique.

## 6. The reduced solve, at an unseen parameter

Algorithm 1, line 4. The property to check is that the reduced multiplier stays
non-negative — that is what the cone construction buys.

In [ ]:
Theta = pod_basis(S_pri, n_modes=8)
mu_new = np.array([0.95, 0.28])
gap = obstacle_gap(60, mu_new)
f = F[:, 0]

u_hf, lam_hf = solve_hf(A, f, gap)
u_rb, lam_rb = solve_reduced(A, f, gap, Theta, res.generators)

print(f"relative primal error : {np.linalg.norm(u_rb-u_hf)/np.linalg.norm(u_hf):.4e}")
print(f"relative dual error   : {np.linalg.norm(lam_rb-lam_hf)/np.linalg.norm(lam_hf):.4e}")
print(f"min reduced multiplier: {lam_rb.min():.3e}   <- must be >= 0")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(u_hf, label="HF"); axes[0].plot(u_rb, "--", label="RB")
axes[0].plot(gap, ":", c="grey", label="obstacle"); axes[0].set_title("primal u")
axes[1].plot(lam_hf, label="HF"); axes[1].plot(lam_rb, "--", label="RB")
axes[1].set_title(r"dual $\lambda$")
for a in axes: a.legend(fontsize=8); a.grid(alpha=.3)
plt.tight_layout()